# Model Training
#### 1.1 Import Data and Required Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    BaggingClassifier, 
    GradientBoostingClassifier, 
    AdaBoostClassifier
    )
from sklearn.metrics import (accuracy_score, 
                             precision_score, 
                             recall_score, 
                             f1_score, 
                             classification_report
                             )

In [2]:
df = pd.read_csv("G:/Graduation Project/Fertilizer_Recommendation_System/notebook/data/fertilizer_recommendation_dataset.csv")
df.head(3)

,Temperature,Moisture,Rainfall,PH,Nitrogen,Phosphorous,Potassium,Carbon,Soil,Crop,Fertilizer,Remark
0,50.179845,0.725893,205.600816,6.227358,66.701872,76.963560,96.429065,0.496300,Loamy Soil,rice,Compost,Enhances organic matter and improves soil stru...
1,21.633318,0.721958,306.081601,7.173131,71.583316,163.057636,148.128347,1.234242,Loamy Soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
2,23.060964,0.685751,259.336414,7.380793,75.709830,62.091508,80.308971,1.795650,Peaty Soil,rice,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...


#### Preparing X and Y variables

In [3]:
X = df.drop(columns=['Fertilizer', 'Remark'],axis=1)
y = df['Fertilizer']

In [4]:
X.head()

,Temperature,Moisture,Rainfall,PH,Nitrogen,Phosphorous,Potassium,Carbon,Soil,Crop
0,50.179845,0.725893,205.600816,6.227358,66.701872,76.963560,96.429065,0.496300,Loamy Soil,rice
1,21.633318,0.721958,306.081601,7.173131,71.583316,163.057636,148.128347,1.234242,Loamy Soil,rice
2,23.060964,0.685751,259.336414,7.380793,75.709830,62.091508,80.308971,1.795650,Peaty Soil,rice
3,26.241975,0.755095,212.703513,6.883367,78.033687,151.012521,153.005712,1.517556,Loamy Soil,rice
4,21.490157,0.730672,268.786767,7.578760,71.765123,66.257371,97.000886,1.782985,Peaty Soil,rice


In [5]:
y.head()

0                       Compost
1       Balanced NPK Fertilizer
2    Water Retaining Fertilizer
3       Balanced NPK Fertilizer
4            Organic Fertilizer
Name: Fertilizer, dtype: object

In [6]:
print(f"Categories in 'Soil' variable: {df['Soil'].unique()}")
print(f"Categories in 'Crop' variable: {df['Crop'].unique()}")

Categories in 'Soil' variable: ['Loamy Soil' 'Peaty Soil' 'Acidic Soil' 'Neutral Soil' 'Alkaline Soil']
Categories in 'Crop' variable: ['rice' 'wheat' 'Mung Bean' 'Tea' 'millet' 'maize' 'Lentil' 'Jute'
 'Coffee' 'Cotton' 'Ground Nut' 'Peas' 'Rubber' 'Sugarcane' 'Tobacco'
 'Kidney Beans' 'Moth Beans' 'Coconut' 'Black gram' 'Adzuki Beans'
 'Pigeon Peas' 'Chickpea' 'banana' 'grapes' 'apple' 'mango' 'muskmelon'
 'orange' 'papaya' 'pomegranate' 'watermelon' 'potatoes' 'tomatoes'
 'berseem clover']


#### Pipeline for preprocessing

In [7]:
num_features = X.select_dtypes(exclude='O').columns
cat_features = X.select_dtypes(include='O').columns

num_trans = RobustScaler()
cat_trans = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer([
    ("OneHotEncoder", cat_trans, cat_features),
    ("RobustScaler", num_trans, num_features)
])

In [8]:
# Encoded y (df['label'])
le = LabelEncoder()
y_encoded = le.fit_transform(y)

y_encoded

array([1, 0, 9, ..., 2, 4, 2])

#### Split data into train and test

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((2695, 10), (674, 10))

In [10]:
# Create scaler
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

#### Create an Evaluate Function to give all metrics after model Training

In [11]:
def evaluate_classification_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted, average='weighted', zero_division=0)
    recall = recall_score(true, predicted, average='weighted', zero_division=0)
    f1 = f1_score(true, predicted, average='weighted', zero_division=0)
    return accuracy, precision, recall, f1

#### Modeling

In [12]:
models = {
    'LogisticRegression': LogisticRegression(),
    'GaussianNB':GaussianNB(),
    'SVC':SVC(),
    'KNeighborsClassifier':KNeighborsClassifier(),
    'DecisionTreeClassifier':DecisionTreeClassifier(),
    'ExtraTreeClassifier':ExtraTreeClassifier(),
    'RandomForestClassifier':RandomForestClassifier(),
    'BaggingClassifier':BaggingClassifier(),
    'GradientBoostingClassifier':GradientBoostingClassifier(),
    'AdaBoostClassifier':AdaBoostClassifier()
}

model_list = []
accuracy_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_accur , model_train_prec, model_train_recall, model_train_f1 = evaluate_classification_model(y_train, y_train_pred)
    model_test_accur , model_test_prec, model_test_recall, model_test_f1 = evaluate_classification_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(model_train_accur))
    print("- F1 Score: {:.4f}".format(model_train_f1))
    print("- Precision: {:.4f}".format(model_train_prec))
    print("- Recall: {:.4f}".format(model_train_recall))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Accuracy: {:.4f}".format(model_test_accur))
    print("- F1 Score: {:.4f}".format(model_test_f1))
    print("- Precision: {:.4f}".format(model_test_prec))
    print("- Recall: {:.4f}".format(model_test_recall))

    accuracy_list.append(model_test_accur)
    
    print('='*35)
    print('\n')

LogisticRegression
Model performance for Training set
- Accuracy: 0.8445
- F1 Score: 0.8408
- Precision: 0.8412
- Recall: 0.8445
----------------------------------
Model performance for Test set
- Accuracy: 0.8220
- F1 Score: 0.8171
- Precision: 0.8186
- Recall: 0.8220


GaussianNB
Model performance for Training set
- Accuracy: 0.1737
- F1 Score: 0.1243
- Precision: 0.4861
- Recall: 0.1737
----------------------------------
Model performance for Test set
- Accuracy: 0.1662
- F1 Score: 0.1237
- Precision: 0.5310
- Recall: 0.1662


SVC
Model performance for Training set
- Accuracy: 0.8790
- F1 Score: 0.8721
- Precision: 0.8855
- Recall: 0.8790
----------------------------------
Model performance for Test set
- Accuracy: 0.8116
- F1 Score: 0.7985
- Precision: 0.8275
- Recall: 0.8116


KNeighborsClassifier
Model performance for Training set
- Accuracy: 0.7751
- F1 Score: 0.7651
- Precision: 0.7786
- Recall: 0.7751
----------------------------------
Model performance for Test set
- Accuracy

#### View Results

In [13]:
results_df = pd.DataFrame({
    'Model': model_list,
    'Test Accuracy': accuracy_list
}).sort_values(by='Test Accuracy', ascending=False)

print("--- Model Comparison ---")
results_df

--- Model Comparison ---


,Model,Test Accuracy
8,GradientBoostingClassifier,0.998516
4,DecisionTreeClassifier,0.991098
7,BaggingClassifier,0.991098
6,RandomForestClassifier,0.980712
0,LogisticRegression,0.821958
2,SVC,0.811573
9,AdaBoostClassifier,0.663205
3,KNeighborsClassifier,0.649852
5,ExtraTreeClassifier,0.580119
1,GaussianNB,0.166172


**insight**
- GradientBoostingClassifier is the winner!!

In [14]:
GradientBoost = GradientBoostingClassifier()
GradientBoost = GradientBoost.fit(X_train, y_train)
y_pred = GradientBoost.predict(X_test)
score = accuracy_score(y_test, y_pred)*100
print(f" Accuracy of the model is {score:.2f}%")

 Accuracy of the model is 99.85%


#### Difference between Actual and Predicted Values

In [15]:
pred_df = pd.DataFrame({
    'Actual Value':y_test,
    'Predicted Value':y_pred,
    'Difference':y_test-y_pred
    })
pred_df

,Actual Value,Predicted Value,Difference
0,6,6,0
1,9,9,0
2,1,1,0
3,1,1,0
4,6,6,0
...,...,...,...
669,2,2,0
670,2,2,0
671,6,6,0
672,9,9,0
